# Лабораторна робота 9. Тонке налаштування (Fine Tuning): VGG16

Ноутбук 1 з 3. Виконав: Бабин Богдан Олегович, група 443-4

**VGG16** — 16 шарів: 13 згорткових (фільтри 3×3) у 5 блоках + 3 повнозв’язні; ~14,7 млн параметрів у згортковій частині. Мережа попередньо навчена на ImageNet.

План:
1. Підготовка даних скриптом з лабораторної 7.
2. **Перенесення навчання** (як у лабі 8): згорткова частина заморожена, навчаємо лише класифікатор.
   Гіперпараметри класифікатора підбираємо на **заздалегідь обчислених ознаках** — згорткова частина заморожена,
   тож її вихід для кожного зображення завжди однаковий, і його можна порахувати один раз. Це дає ~30 експериментів за кілька хвилин.
3. **Тонке налаштування**: розморожуємо 1 або 2 останні блоки згорткової частини і донавчаємо їх разом із класифікатором
   з дуже малою швидкістю навчання.
4. Порівняння з лабораторною 8, розпізнавання власних фото, збереження найкращої моделі.

**Перед запуском:** Среда выполнения → T4 GPU → «Выполнить все».

In [ ]:
import os, shutil, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import load_img, img_to_array
from google.colab import files
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
%matplotlib inline

tf.keras.utils.set_random_seed(42)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Набір даних Dogs vs Cats і скрипт підготовки (з лабораторних 7–8)
Набір https://www.kaggle.com/c/dogs-vs-cats (25 000 фото) завантажуємо з офіційного дзеркала Microsoft (Kaggle вимагає акаунт).
Скрипт відкидає пошкоджені файли, перемішує і ділить на train / val / test (70 / 15 / 15); беремо 3 000 зображень кожного класу,
як у лабораторній 8, щоб результати можна було порівняти.

In [ ]:
!wget -q -O cats_dogs.zip https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -q -o cats_dogs.zip
!ls

In [ ]:
%%writefile prepare_dataset.py
"""
prepare_dataset.py — підготовка набору Dogs vs Cats для навчання нейромережі в Keras.

1) знаходить усі зображення котів і собак (підтримує обидва формати архіву:
   PetImages/Cat, PetImages/Dog  — дзеркало Microsoft,  або  train/cat.N.jpg, train/dog.N.jpg — Kaggle);
2) відкидає пошкоджені файли (у наборі є порожні та «биті» зображення);
3) перемішує і ділить на навчальний (70%), оціночний (15%) і тестовий (15%) набори;
4) копіює у структуру каталогів, яку розуміє ImageDataGenerator.flow_from_directory:
      dataset/train/cats, dataset/train/dogs, dataset/val/..., dataset/test/...
"""
import os, sys, shutil, random
from PIL import Image

dataset_dir = 'dataset'                         # куди складаємо результат
train_dir = os.path.join(dataset_dir, 'train')  # каталог з даними для навчання
val_dir = os.path.join(dataset_dir, 'val')      # каталог з даними для перевірки
test_dir = os.path.join(dataset_dir, 'test')    # каталог з даними для тестування
val_data_portion = 0.15                         # частка зображень для перевірки
test_data_portion = 0.15                        # частка зображень для тестування
random.seed(42)
# необов'язковий аргумент: максимум зображень кожного класу (для навчання без GPU), напр. python prepare_dataset.py 3000
max_per_class = int(sys.argv[1]) if len(sys.argv) > 1 and sys.argv[1].isdigit() else None


def find_images():
    """Повертає словник {'cats': [шляхи], 'dogs': [шляхи]}."""
    if os.path.isdir('PetImages'):              # дзеркало Microsoft (той самий набір Kaggle)
        lst = lambda d: [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith('.jpg')]
        return {'cats': lst('PetImages/Cat'), 'dogs': lst('PetImages/Dog')}
    if os.path.isdir('train'):                  # оригінальний train.zip з Kaggle: cat.0.jpg, dog.0.jpg ...
        files = os.listdir('train')
        return {'cats': [os.path.join('train', f) for f in files if f.startswith('cat.')],
                'dogs': [os.path.join('train', f) for f in files if f.startswith('dog.')]}
    raise FileNotFoundError('Не знайдено ні PetImages/, ні train/ — розпакуйте архів з набором даних')


def is_valid(path):
    """Перевіряє, що файл — справжнє зображення, яке відкривається."""
    try:
        if os.path.getsize(path) == 0:
            return False
        with Image.open(path) as img:
            img.verify()                        # перевірка цілісності файлу
        with Image.open(path) as img:
            img.convert('RGB').load()           # повне декодування
        return True
    except Exception:
        return False


def create_directory(dir_name):
    """Створює каталог з підкаталогами cats і dogs (старий видаляє)."""
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)
    os.makedirs(os.path.join(dir_name, 'cats'))
    os.makedirs(os.path.join(dir_name, 'dogs'))


def copy_images(files, dest_dir, class_name):
    for i, src in enumerate(files):
        shutil.copy2(src, os.path.join(dest_dir, class_name, f'{class_name[:-1]}.{i}.jpg'))


images = find_images()
for cls in images:
    total = len(images[cls])
    images[cls] = [p for p in images[cls] if is_valid(p)]
    print(f'{cls}: знайдено {total}, пошкоджених відкинуто {total - len(images[cls])}')

# однакова кількість зображень кожного класу
nb_images = min(len(images['cats']), len(images['dogs']))
if max_per_class:
    nb_images = min(nb_images, max_per_class)
    print(f'Обмеження: {nb_images} зображень кожного класу')
start_val_data_idx = int(nb_images * (1 - val_data_portion - test_data_portion))
start_test_data_idx = int(nb_images * (1 - test_data_portion))

for d in (train_dir, val_dir, test_dir):
    create_directory(d)

for cls, files in images.items():
    random.shuffle(files)
    files = files[:nb_images]
    copy_images(files[:start_val_data_idx], train_dir, cls)
    copy_images(files[start_val_data_idx:start_test_data_idx], val_dir, cls)
    copy_images(files[start_test_data_idx:], test_dir, cls)

for d in (train_dir, val_dir, test_dir):
    print(f'{d}: cats = {len(os.listdir(os.path.join(d, "cats")))}, dogs = {len(os.listdir(os.path.join(d, "dogs")))}')

In [ ]:
!python prepare_dataset.py 3000

In [ ]:
img_width, img_height = 150, 150
input_shape = (img_width, img_height, 3)
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)   # обробка як при навчанні VGG16 на ImageNet

def flow(d, shuffle, bs=32):
    return datagen.flow_from_directory('dataset/' + d, target_size=(img_width, img_height),
                                       batch_size=bs, class_mode='binary', shuffle=shuffle)
train_generator, val_generator, test_generator = flow('train', True), flow('val', False), flow('test', False)
print(train_generator.class_indices)

## 2. Попередньо навчена мережа VGG16
`include_top=False` — лише згорткова частина. Для тонкого налаштування розморожуватимемо останні блоки: **1 блок = block5 (3 згорткові шари), 2 блоки = block4 + block5**.

In [ ]:
base_net = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
base_net.trainable = False
names = [l.name for l in base_net.layers]
CUTS = {1: 'block4_pool', 2: 'block3_pool'}   # розморожуються всі шари ПІСЛЯ цього шару
print('Шарів:', len(base_net.layers), '| вихід:', base_net.output_shape, f'| параметрів: {base_net.count_params():,}')
for k, cut in CUTS.items():
    layers = base_net.layers[names.index(cut) + 1:]
    n_par = sum(int(np.prod(w.shape)) for l in layers if not isinstance(l, BatchNormalization) for w in l.weights)
    print(f'{k} блок(и): шари {layers[0].name} … {layers[-1].name} ({len(layers)} шарів, {n_par:,} параметрів)')

## 3. Перенесення навчання + підбір гіперпараметрів класифікатора

### 3.1. Ознаки з замороженої згорткової частини
Пропускаємо всі зображення через VGG16 + Flatten (карта ознак 4×4×512 → 8192), як у лекції один раз.

In [ ]:
feat_net = Sequential([Input(shape=input_shape), base_net, Flatten()])
t0 = time.time()
X_tr = feat_net.predict(flow('train', False), verbose=0); y_tr = flow('train', False).classes
X_va = feat_net.predict(val_generator, verbose=0);         y_va = val_generator.classes
X_te = feat_net.predict(test_generator, verbose=0);        y_te = test_generator.classes
print('Ознаки:', X_tr.shape, X_va.shape, X_te.shape, f'| {time.time() - t0:.0f} с')

### 3.2. Експерименти (Exercises із завдання)
Змінюємо по одному гіперпараметру, решту беремо найкращими з попередніх кроків (жадібний пошук).
Початкові значення — з лабораторної 8: Dense 256, Dropout 0.5, без додаткового шару, batch 32, Adam 1e-4, 10 епох.
Критерій — середня точність на **перевірочних** даних за 3 останні епохи (тестові дані для вибору не використовуємо).

| група | значення |
|---|---|
| нейронів у Dense | 32, 128, 256, 512, 1024 |
| Dropout | 0.25, 0.5, 0.6, 0.8 |
| додатковий Dense перед виходом | немає, 32, 128, 256 |
| розмір міні-вибірки | 25, 32, 50, 100, 125 |
| оптимізатор (стандартна швидкість) | sgd (0.01), adam (0.001), rmsprop (0.001) |
| швидкість навчання Adam | 1e-3, 1e-4, 1e-5, 1e-6 |
| кількість епох | 5, 10, 15, 20, 25 |

In [ ]:
def make_head(cfg, dim):
    layers = [Input(shape=(dim,)), Dense(cfg['units'], activation='relu'), Dropout(cfg['dropout'])]
    if cfg['extra']:
        layers += [Dense(cfg['extra'], activation='relu')]
    layers += [Dense(1, activation='sigmoid')]
    return Sequential(layers, name='classifier')

def make_opt(name, lr):
    cls = {'adam': Adam, 'rmsprop': RMSprop, 'sgd': SGD}[name]
    return cls() if lr is None else cls(learning_rate=lr)

def train_head(cfg, epochs=None):
    tf.keras.utils.set_random_seed(42)
    head = make_head(cfg, X_tr.shape[1])
    head.compile(loss='binary_crossentropy', optimizer=make_opt(cfg['opt'], cfg['lr']), metrics=['accuracy'])
    h = head.fit(X_tr, y_tr, batch_size=cfg['batch'], epochs=epochs or cfg['epochs'],
                 validation_data=(X_va, y_va), verbose=0)
    return head, h.history

def score(hist, upto=None):
    v = hist['val_accuracy'][:upto] if upto else hist['val_accuracy']
    return float(np.mean(v[-3:]))

cfg = dict(units=256, dropout=0.5, extra=0, batch=32, opt='adam', lr=1e-4, epochs=10)
GRID = [('units', [32, 128, 256, 512, 1024]), ('dropout', [0.25, 0.5, 0.6, 0.8]), ('extra', [0, 32, 128, 256]),
        ('batch', [25, 32, 50, 100, 125]), ('opt', ['sgd', 'adam', 'rmsprop']), ('lr', [1e-3, 1e-4, 1e-5, 1e-6])]
GROUP = {'units': 'нейронів у Dense', 'dropout': 'Dropout', 'extra': 'додатковий Dense', 'batch': 'міні-вибірка',
         'opt': 'оптимізатор', 'lr': 'швидкість Adam', 'epochs': 'епох'}
rows, t0, cur_s = [], time.time(), -1
for key, values in GRID:
    best_v, best_s = (None, cur_s) if key == 'lr' else (None, -1)   # Adam з іншою швидкістю має перевершити найкращий оптимізатор
    for v in values:
        c = dict(cfg, **{key: v})
        if key == 'opt': c['lr'] = None                      # стандартна швидкість кожного оптимізатора
        if key == 'lr': c['opt'] = 'adam'
        _, hist = train_head(c)
        s = score(hist)
        rows.append({'група': GROUP[key], 'значення': v, 'train acc, %': round(hist['accuracy'][-1] * 100, 2),
                     'val acc, %': round(hist['val_accuracy'][-1] * 100, 2), 'val acc (3 ост.), %': round(s * 100, 2),
                     'val loss': round(hist['val_loss'][-1], 4)})
        if s > best_s: best_v, best_s = v, s
    if key == 'lr':
        if best_v is not None: cfg.update(opt='adam', lr=best_v)
    else:
        cfg[key] = best_v
        if key == 'opt' and best_v != 'adam': cfg['lr'] = None
    cur_s = best_s
    print(f'{GROUP[key]:>18}: найкраще {cfg["lr"] if key == "lr" else best_v}  (val acc {best_s*100:.2f}%)')

# кількість епох: одне навчання на 25 епох, точність після 5, 10, … епох
_, hist25 = train_head(cfg, epochs=25)
marks = [e for e in [5, 10, 15, 20, 25] if e <= len(hist25['val_accuracy'])] or [len(hist25['val_accuracy'])]
for e in marks:
    rows.append({'група': 'епох', 'значення': e, 'train acc, %': round(hist25['accuracy'][e - 1] * 100, 2),
                 'val acc, %': round(hist25['val_accuracy'][e - 1] * 100, 2), 'val acc (3 ост.), %': round(score(hist25, e) * 100, 2),
                 'val loss': round(hist25['val_loss'][e - 1], 4)})
cfg['epochs'] = max(marks, key=lambda e: score(hist25, e))
print(f'{"епох":>18}: найкраще {cfg["epochs"]}')
print(f'\nЧас пошуку: {time.time() - t0:.0f} с\nНайкращі гіперпараметри: {cfg}')

In [ ]:
hp = pd.DataFrame(rows)
hp.insert(0, 'модель', 'VGG16')
hp.to_csv('lab9_VGG16_hyperparams.csv', index=False)
pd.set_option('display.max_rows', 100)
hp

In [ ]:
groups = list(dict.fromkeys(hp['група']))
fig, axes = plt.subplots(2, 4, figsize=(18, 7)); axes = axes.ravel()
for ax, g in zip(axes, groups):
    d = hp[hp['група'] == g]
    vals = d['val acc (3 ост.), %'].values
    bars = ax.bar(d['значення'].astype(str), vals, color=['tab:green' if v == vals.max() else 'tab:blue' for v in vals])
    ax.set_ylim(max(50, vals.min() - 2), min(100, vals.max() + 1)); ax.set_title(g); ax.grid(axis='y', alpha=.3)
    for b, v in zip(bars, vals): ax.text(b.get_x() + b.get_width() / 2, v, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
ax = axes[len(groups)]
ax.plot(range(1, len(hist25['accuracy']) + 1), hist25['accuracy'], 'b-o', ms=3, label='Навчальні')
ax.plot(range(1, len(hist25['val_accuracy']) + 1), hist25['val_accuracy'], 'r-o', ms=3, label='Перевірочні')
ax.set_title('25 епох: точність'); ax.set_xlabel('Епоха'); ax.legend(); ax.grid(alpha=.3)
plt.suptitle('VGG16: підбір гіперпараметрів класифікатора (точність на перевірці, %)'); plt.tight_layout(); plt.show()

### 3.3. Модель перенесення навчання з найкращими гіперпараметрами
Навчений на ознаках класифікатор ставимо поверх замороженої згорткової частини — виходить повна модель
(та сама, що в лабораторній 8, але з підібраними гіперпараметрами).

In [ ]:
def plot_history(h, title):
    ep = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(13, 4))
    plt.subplot(1, 2, 1); plt.plot(ep, h['accuracy'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_accuracy'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Точність'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.subplot(1, 2, 2); plt.plot(ep, h['loss'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_loss'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Помилка (loss)'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(title); plt.tight_layout(); plt.show()

head, tl_hist = train_head(cfg)
model = Sequential([Input(shape=input_shape), base_net, Flatten(), head])
model.compile(loss='binary_crossentropy', optimizer=Adam(1e-5), metrics=['accuracy'])
tl_loss, tl_acc = model.evaluate(test_generator, verbose=0)
plot_history(tl_hist, f'VGG16: перенесення навчання (згорткова частина заморожена) — test acc {tl_acc*100:.2f}%')
model.summary()
print(f'Перенесення навчання: test acc = {tl_acc*100:.2f}%, test loss = {tl_loss:.4f}  (лаба 8: 97.56%)')
model.save_weights('tl.weights.h5')

## 4. Тонке налаштування (Fine Tuning)
1. Беремо модель після перенесення навчання (класифікатор уже навчений — інакше великі випадкові градієнти
   від нового класифікатора зруйнували б ваги ImageNet).
2. **Розморожуємо** 1 або 2 останні блоки згорткової частини (1 блок = block5 (3 згорткові шари), 2 блоки = block4 + block5).
   Шари BatchNormalization залишаємо замороженими (режим inference), як рекомендує документація Keras.
3. **Донавчаємо** розморожені шари разом із класифікатором з дуже малою швидкістю навчання (Adam 1e-5 і інші),
   щоб лише трохи підлаштувати ознаки під котів і собак. EarlyStopping (patience 3) повертає ваги найкращої епохи.

Кожен експеримент починається з тих самих ваг після перенесення навчання.

In [ ]:
def unfreeze(k):
    base_net.trainable = True
    cut = names.index(CUTS[k]) if k else len(names)
    for i, layer in enumerate(base_net.layers):
        layer.trainable = i > cut and not isinstance(layer, BatchNormalization)

FT_RUNS = [(1, 1e-5), (2, 1e-5), (1, 1e-4), (1, 1e-6)]   # (розморожених блоків, швидкість навчання Adam)
FT_EPOCHS = 10
ft_rows, ft_hist, best_w, best_acc = [], {}, None, tl_acc
for k, lr in FT_RUNS:
    print('=' * 70, f'\nРозморожено блоків: {k}, Adam lr = {lr}')
    model.load_weights('tl.weights.h5')
    unfreeze(k)
    model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=lr), metrics=['accuracy'])
    n_tr = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    tf.keras.utils.set_random_seed(42); t0 = time.time()
    h = model.fit(train_generator, epochs=FT_EPOCHS, validation_data=val_generator, verbose=2,
                  callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])
    t = time.time() - t0
    loss, acc = model.evaluate(test_generator, verbose=0)
    hh = h.history; best_ep = int(np.argmin(hh['val_loss']))
    ft_rows.append({'блоків': k, 'lr': lr, 'навчаються параметрів': n_tr, 'епох': len(hh['loss']), 'найкраща епоха': best_ep + 1,
                    'train acc, %': round(hh['accuracy'][best_ep] * 100, 2), 'val acc, %': round(hh['val_accuracy'][best_ep] * 100, 2),
                    'val loss': round(hh['val_loss'][best_ep], 4), 'test acc, %': round(acc * 100, 2), 'test loss': round(loss, 4), 'час, с': round(t)})
    ft_hist[(k, lr)] = hh
    plot_history(hh, f'VGG16: fine tuning, {k} блок(и), lr={lr} — test acc {acc*100:.2f}%')
    if acc > best_acc:
        best_acc, best_w, best_run = acc, model.get_weights(), (k, lr)

In [ ]:
ft = pd.DataFrame(ft_rows)
ft.insert(0, 'модель', 'VGG16')
ft.to_csv('lab9_VGG16_finetune.csv', index=False)
ft

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for (k, lr), h in ft_hist.items():
    e = range(1, len(h['val_accuracy']) + 1)
    ax[0].plot(e, h['val_accuracy'], '-o', ms=3, label=f'{k} бл., lr={lr}')
    ax[1].plot(e, np.array(h['accuracy']) - np.array(h['val_accuracy']), '-o', ms=3, label=f'{k} бл., lr={lr}')
ax[0].axhline(tl_hist['val_accuracy'][-1], color='k', ls='--', lw=1, label='без fine tuning')
ax[0].set_title('Точність на перевірочних даних'); ax[1].set_title('Розрив train acc − val acc (перенавчання)')
for a_ in ax: a_.set_xlabel('Епоха'); a_.legend(fontsize=8); a_.grid(alpha=.3)
ax[1].axhline(0, color='k', lw=.5)
plt.tight_layout(); plt.show()

## 5. Найкраща модель і порівняння з лабораторною 8

In [ ]:
if best_w is not None:
    unfreeze(best_run[0]); model.set_weights(best_w)
    best_name = f'fine tuning ({best_run[0]} блок(и), lr={best_run[1]})'
else:
    unfreeze(0); model.load_weights('tl.weights.h5')
    best_name = 'перенесення навчання (fine tuning не покращив)'
model.compile(loss='binary_crossentropy', optimizer=Adam(1e-5), metrics=['accuracy'])
best_model = model
best_loss, best_acc = best_model.evaluate(test_generator, verbose=0)
best_model.save('lab9_VGG16_best.keras')
print(f'Найкраща модель: {best_name} | test acc = {best_acc*100:.2f}% | файл {os.path.getsize("lab9_VGG16_best.keras") / 2**20:.1f} МБ')
cmp = pd.DataFrame([
    {'етап': 'лаба 8: перенесення навчання (Dense 256, Dropout 0.5, Adam 1e-4, 8 епох)', 'test acc, %': 97.56},
    {'етап': 'лаба 9: перенесення навчання + підібрані гіперпараметри', 'test acc, %': round(tl_acc * 100, 2)},
] + [{'етап': f'лаба 9: fine tuning, {r["блоків"]} блок(и), lr={r["lr"]}', 'test acc, %': r['test acc, %']} for r in ft_rows])
cmp

## 6. Розпізнавання нових зображень
Ті самі власні фото, що в лабораторних 7–8: 20 звичайних і 6 «складних» (тварина у високій траві).

In [ ]:
os.makedirs('test_images', exist_ok=True)
# архів lab8_test_images.zip завантажено через панель «Файли»; якщо його немає — відкриється вікно files.upload()
if os.path.exists('lab8_test_images.zip'):
    uploaded = {'lab8_test_images.zip': None}
else:
    uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('test_images')
    else:
        shutil.move(name, os.path.join('test_images', name))
paths = sorted(os.path.join(r, f) for r, _, fs in os.walk('test_images') for f in fs
               if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('._'))
print(len(paths), 'зображень')

In [ ]:
def predict_images(paths, title):
    rows = []
    cols = 6; rows_n = int(np.ceil(len(paths) / cols))
    plt.figure(figsize=(cols * 2.8, rows_n * 3.1))
    for i, p in enumerate(paths):
        x = img_to_array(load_img(p, target_size=(img_width, img_height)))
        x = preprocess_input(np.expand_dims(x, axis=0))
        prob = float(best_model.predict(x, verbose=0)[0][0])
        answer = 'собака' if prob > 0.5 else 'кіт'
        name = os.path.basename(p).lower()
        truth = 'кіт' if name.startswith('cat') else ('собака' if name.startswith('dog') else None)
        ok = None if truth is None else answer == truth
        conf = prob if prob > 0.5 else 1 - prob
        plt.subplot(rows_n, cols, i + 1); plt.imshow(load_img(p)); plt.axis('off')
        plt.title(f'{answer} ({conf*100:.0f}%)\n{os.path.basename(p)}', fontsize=8,
                  color='black' if ok is None else ('green' if ok else 'red'))
        rows.append({'файл': os.path.basename(p), 'справжній клас': truth or '?', 'відповідь': answer,
                     'впевненість, %': round(conf * 100, 1), 'вірно': ok})
    plt.suptitle(title); plt.tight_layout(); plt.show()
    df = pd.DataFrame(rows)
    k = df[df['вірно'].notna()]
    print(f'{title}: вірно {int(k["вірно"].sum())} з {len(k)}')
    return df

normal = [p for p in paths if '_h' not in os.path.basename(p)]
hard = [p for p in paths if '_h' in os.path.basename(p) and '_crop' not in os.path.basename(p)]
own = predict_images(normal, 'VGG16 (лаба 9): звичайні зображення')
own_hard = predict_images(hard, 'VGG16 (лаба 9): складні зображення (тварина в траві)') if hard else pd.DataFrame()
pd.concat([own, own_hard]).to_csv('lab9_VGG16_own.csv', index=False)
pd.concat([own, own_hard])

In [ ]:
summary = pd.DataFrame([{'модель': 'VGG16', 'гіперпараметри': str(cfg), 'найкраща модель': best_name,
    'test acc лаба 8, %': 97.56, 'test acc перенесення, %': round(tl_acc * 100, 2), 'test acc найкращої, %': round(best_acc * 100, 2),
    'власні фото': f"{int(own['вірно'].sum())}/{len(own)} (лаба 8: 19/20)",
    'складні фото': (f"{int(own_hard['вірно'].sum())}/{len(own_hard)}" if len(own_hard) else '—') + ' (лаба 8: 5/6)'}])
summary.to_csv('lab9_VGG16_summary.csv', index=False)
summary.T

In [ ]:
for f in ['lab9_VGG16_best.keras', 'lab9_VGG16_hyperparams.csv', 'lab9_VGG16_finetune.csv', 'lab9_VGG16_own.csv', 'lab9_VGG16_summary.csv']:
    files.download(f)